# Notebook F — Verdict critique : QMKL en 2026

Ce notebook pose les questions difficiles : qu'est-ce que QMKL peut et ne peut pas faire ? Nos résultats sont-ils une victoire ou un échec ? Quand faut-il y revenir ?

**Plan :**
1. Ce que QMKL peut théoriquement mieux faire
2. Ce que QMKL ne peut PAS faire (limites dures)
3. Analyse honnête des résultats du projet
4. Guide de décision visuel
5. Perspectives 2026–2030
6. Synthèse : les 5 vérités inconfortables sur QMKL


## Section 1 — Ce que QMKL peut théoriquement mieux faire

### Avantage 1 : Richesse de l'espace de représentation

Le RKHS associé à $K_Z$ ou $K_{ZZ}$ vit dans un espace de dimension $2^Q$ (exponentiel en $Q$).
En comparaison, un kernel polynomial de degré $p$ sur $\mathbb{R}^d$ vit dans $\binom{d+p}{p} \sim d^p/p!$ dimensions.

Pour $Q=6, d=30, p=3$ : Quantique = 64 dimensions vs Polynomial = 4960 dimensions — le polynomial est plus grand ici.
Pour $Q=20, d=30, p=3$ : Quantique = $10^6$ dimensions vs Polynomial = 4960 dimensions — le quantique domine.

**Mais :** simuler un tel circuit prend $O(2^{20}) = 10^6$ opérations par évaluation de kernel. Le gain en expressivité est annulé par le coût de simulation.

### Avantage 2 : Géométrie différente du RBF

$K_{ZZ}(x, x') = \prod_k \cos^2(\alpha(x_k - x'_k)) \cdot \prod_{k<l} \cos^2(\alpha(x_k x_l - x'_k x'_l))$

Le terme $x_k x_l - x'_k x'_l$ est différent de $(x_k - x'_k)(x_l - x'_l)$.
C'est une géométrie **non-euclidienne** dans l'espace des produits de features.

Pour des données où cette géométrie est naturelle (e.g., features qui interagissent multiplicativement), $K_{ZZ}$ peut capturer des structures que le RBF manque.

### Avantage 3 : Formulation QUBO pour la sélection

QUBO est un framework d'optimisation combinatoire directement compatible avec les **annealers quantiques futurs**.
La formulation est indépendante de la simulation quantique — elle peut être exécutée sur D-Wave ou IBM QAOA même avec les QPU actuels.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

def K_ZZ(X, alpha=1.0):
    n,d=X.shape; K=np.ones((n,n))
    for k in range(d):
        diff=X[:,k:k+1]-X[:,k].reshape(1,-1); K*=np.cos(alpha*diff)**2
    for k in range(d):
        for l in range(k+1,d):
            c=X[:,k]*X[:,l]; diff=c.reshape(-1,1)-c.reshape(1,-1)
            K*=np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq=np.sum(X**2,axis=1,keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

def eval_K(K, y, cv=5):
    K=K.copy(); eigs=np.linalg.eigvalsh(K)
    if eigs.min()<0: K+=(-eigs.min()+1e-8)*np.eye(len(K))
    try: return cross_val_score(SVC(kernel='precomputed',C=1.),K,y,cv=cv,scoring='roc_auc').mean()
    except: return np.nan

# ── Démontrer l'avantage de ZZ sur les données XOR multiplicatif ─────────
n = 120
X_xor = np.random.uniform(-1.5, 1.5, (n, 2))
y_xor = np.where(X_xor[:,0] * X_xor[:,1] > 0.1, 1, -1)

# Préparer les données
Xp = MinMaxScaler(feature_range=(0,2)).fit_transform(X_xor)
Xp = np.hstack([Xp, np.zeros((n, 2))])  # pad à Q=4

K_zz = K_ZZ(Xp, alpha=1.5)
np.fill_diagonal(K_zz, 1.0)
K_rbf1 = K_rbf(Xp[:,:2], gamma=1.0)
K_rbf_fine = K_rbf(Xp[:,:2], gamma=5.0)
K_poly = (Xp[:,:2] @ Xp[:,:2].T + 1)**3

# Comparer
print("Dataset XOR multiplicatif (x₁·x₂ > 0.1):")
for K, name in [(K_rbf1,'RBF γ=1.0'),(K_rbf_fine,'RBF γ=5.0'),(K_poly,'Poly p=3'),(K_zz,'K_ZZ α=1.5')]:
    auc = eval_K(K.copy(), y_xor)
    print(f"  {name}: AUC = {auc:.4f}")

# ── Visualiser les frontières de décision ────────────────────────────────
xx, yy = np.meshgrid(np.linspace(-1.7, 1.7, 80), np.linspace(-1.7, 1.7, 80))
X_grid = np.column_stack([xx.ravel(), yy.ravel()])
X_grid_scaled = MinMaxScaler(feature_range=(0,2)).fit_transform(
    np.vstack([X_xor, X_grid]))
X_train_s = X_grid_scaled[:n]
X_grid_s  = X_grid_scaled[n:]
X_train_s4 = np.hstack([X_train_s, np.zeros((n,2))])
X_grid_s4  = np.hstack([X_grid_s,  np.zeros((len(X_grid_s),2))])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (K_train, name, gamma_or_alpha) in zip(axes, [
    (K_rbf(X_train_s, gamma=1.0), 'RBF SVM (γ=1.0)',  1.0),
    (K_ZZ(X_train_s4, alpha=1.5), 'K_ZZ SVM (α=1.5)', 1.5),
]):
    np.fill_diagonal(K_train, 1.0)
    eigs=np.linalg.eigvalsh(K_train)
    if eigs.min()<0: K_train+=(-eigs.min()+1e-8)*np.eye(n)
    svm = SVC(kernel='precomputed', C=1., probability=True)
    svm.fit(K_train, y_xor)

    # Kernel entre grille et train
    if 'RBF' in name:
        K_pred = K_rbf(np.vstack([X_grid_s, X_train_s]), gamma=1.0)[:len(X_grid_s), len(X_grid_s):]
    else:
        K_full = K_ZZ(np.vstack([X_grid_s4, X_train_s4]), alpha=1.5)
        K_pred = K_full[:len(X_grid_s4), len(X_grid_s4):]

    try:
        Z = svm.predict_proba(K_pred)[:, 1].reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6, vmin=0, vmax=1)
        ax.contour(xx, yy, Z, levels=[0.5], colors='black', lw=2)
    except:
        pass

    ax.scatter(X_xor[y_xor==1,0], X_xor[y_xor==1,1], c='#e74c3c', s=30, zorder=3)
    ax.scatter(X_xor[y_xor==-1,0], X_xor[y_xor==-1,1], c='#3498db', s=30, marker='s', zorder=3)
    ax.set_title(f'{name}
AUC = {eval_K(K_train, y_xor):.4f}', fontweight='bold')
    ax.set_aspect('equal')

plt.suptitle('Frontière de décision : RBF vs K_ZZ sur XOR multiplicatif
'
             'ZZ capture mieux x₁·x₂ grâce aux termes d'interaction', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 2 — Ce que QMKL ne peut PAS faire

### Limite 1 : Pas d'avantage quantique prouvé

**Fait :** Aucun papier n'a prouvé qu'un kernel quantique fournit une fonction impossible à approcher par un kernel classique de façon efficace, pour des données classiques.

**Résultat de Huang et al. (2021) :** Pour des données classiques, il existe toujours un kernel classique qui approche le kernel quantique. L'avantage quantique n'est garanti que pour des données **générées par un processus quantique** (e.g., simulation de molécules, mesures d'un QPU).

**Conséquence :** Sur des données financières tabulaires, il n'y a aucune raison théorique que QMKL domine les classiques.

### Limite 2 : Barren plateaus à grand Q

Comme montré dans les notebooks B et E, la variance du kernel décroît exponentiellement. Au-delà de $Q = 8$, le kernel devient inutilisable en pratique.

### Limite 3 : Coût de simulation

Pour calculer $K(x, x')$ via un simulateur Qiskit :
- $Q = 4$, $N = 100$ → environ $N^2/2 = 5000$ circuits → ~2s sur CPU
- $Q = 6$, $N = 200$ → $20\,000$ circuits → ~2 min
- $Q = 8$, $N = 500$ → $125\,000$ circuits → ~3h

Le cache kernel sauve pour les expériences répétées, mais le coût initial est élevé.

### Limite 4 : Hardware NISQ bruité

IBM Torino (notre expérience) : le kernel ZZ α=4.0 donne AUC < 0.5 sur hardware réel.
Le bruit des portes CNOT (~0.5% d'erreur par porte) dégrade significativement les circuits profonds.
Correction d'erreurs quantiques (QEC) nécessaire pour une utilisation fiable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)

# ── Coût de simulation en fonction de Q et N ─────────────────────────────
def K_ZZ_time(n_points, Q, n_rep=3):
    '''Mesurer le temps de calcul du kernel ZZ (n_points × Q).'''
    X = np.random.uniform(0, 2, (n_points, Q))
    times = []
    for _ in range(n_rep):
        t0 = time.time()
        n, d = X.shape
        K = np.ones((n, n))
        for k in range(d):
            diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
            K *= np.cos(1.5*diff)**2
        for k in range(d):
            for l in range(k+1, d):
                c = X[:,k]*X[:,l]
                diff = c.reshape(-1,1) - c.reshape(1,-1)
                K *= np.cos(1.5*diff)**2
        times.append(time.time() - t0)
    return np.mean(times)

Qs = [2, 4, 6, 8]
Ns = [50, 100, 150, 200]

# Mesurer les temps
timing_matrix = np.zeros((len(Qs), len(Ns)))
print("Temps de calcul kernel ZZ (secondes):")
print(f"{'Q\\N':5s}", end='')
for N in Ns: print(f"{N:8d}", end='')
print()
for qi, Q in enumerate(Qs):
    print(f"Q={Q:2d}  ", end='')
    for ni, N in enumerate(Ns):
        t = K_ZZ_time(N, Q)
        timing_matrix[qi, ni] = t
        print(f"{t:8.3f}", end='')
    print()

# ── Visualisation de la scalabilité ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
colors = ['#2ecc71','#3498db','#f39c12','#e74c3c']
for qi, (Q, color) in enumerate(zip(Qs, colors)):
    ax.semilogy(Ns, timing_matrix[qi,:], 'o-', color=color, ms=7, lw=2.5, label=f'Q={Q}')

# Courbe N² théorique
N_fit = np.array(Ns, dtype=float)
scale = timing_matrix[0,0] / Ns[0]**2
ax.semilogy(N_fit, scale*N_fit**2, 'k--', lw=1.5, alpha=0.5, label='O(N²) théorique')

# Lignes de référence temporelles
for t_ref, label in [(0.1,'100ms'), (1.0,'1s'), (60.0,'1min'), (3600.,'1h')]:
    ax.axhline(t_ref, color='grey', lw=0.7, ls=':', alpha=0.7)
    ax.text(Ns[-1]+2, t_ref, label, va='center', fontsize=8, color='grey')

ax.set_xlabel('N (taille dataset)', fontsize=11)
ax.set_ylabel('Temps kernel (s)')
ax.set_title('Coût de simulation K_ZZ vs N et Q
(Kernel analytique numpy)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(Ns[0]-5, Ns[-1]+15)
ax.grid(alpha=0.2)

# ── Comparaison résultats projet vs attendus ──────────────────────────────
ax2 = axes[1]

# Résultats du projet QMKL-Finance (valeurs réelles)
datasets_proj = ['German
Credit', 'Bank
Marketing', 'Breast
Cancer']
rbf_auc  = [0.800, 0.875, 0.997]
qmkl_auc = [0.743, 0.861, 0.993]
qubo_auc = [0.823, 0.767, np.nan]  # QUBO seulement sur 2 datasets

x_pos = np.arange(len(datasets_proj))
w = 0.25
b1 = ax2.bar(x_pos - w, rbf_auc, w, label='RBF-SVM', color='#e74c3c', alpha=0.85)
b2 = ax2.bar(x_pos,     qmkl_auc, w, label='QMKL-Centered', color='#3498db', alpha=0.85)
b3 = ax2.bar(x_pos + w, [q if not np.isnan(q) else 0 for q in qubo_auc],
             w, label='QMKL-Boost (QUBO)', color='#2ecc71', alpha=0.85)

# Annotations
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax2.text(bar.get_x()+bar.get_width()/2, h+0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(datasets_proj)
ax2.set_ylabel('AUC')
ax2.set_title('Résultats réels du projet QMKL-Finance
'
              'RBF domine sur les données financières', fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_ylim(0.6, 1.03)
ax2.grid(axis='y', alpha=0.3)

# Déltas
for ds_idx, (rbf, qmkl) in enumerate(zip(rbf_auc, qmkl_auc)):
    diff = qmkl - rbf
    ax2.text(x_pos[ds_idx], 0.62,
             f'{diff:+.3f}', ha='center', fontsize=9, fontweight='bold',
             color='#27ae60' if diff>0 else '#c0392b')

plt.suptitle('Limites de QMKL : coût computationnel et résultats réels',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("
Bilan résultats:")
for ds, rbf, qmkl in zip(datasets_proj, rbf_auc, qmkl_auc):
    sign = '>' if qmkl>rbf else '<'
    print(f"  {ds.replace(chr(10),' ')}: QMKL ({qmkl:.3f}) {sign} RBF ({rbf:.3f}) → "
          f"{'QMKL gagne' if qmkl>rbf else f'RBF gagne de {rbf-qmkl:.3f}'}")


## Section 3 — Guide de décision visuel

### Arbre de décision : faut-il utiliser QMKL ?

```
Données disponibles
      │
      ├── N > 1000 ? ─────── OUI → Non. RBF-SVM + Random Forest.
      │
      └── NON (N ≤ 1000)
            │
            ├── Structure périodique ou interactions multiplicatives ?
            │         │
            │         ├── OUI → QMKL potentiellement utile. Tester ZZ.
            │         │
            │         └── NON → RBF-SVM probablement meilleur.
            │
            ├── Besoin d'interprétabilité de la sélection ?
            │         │
            │         ├── OUI → QUBO (choisit 2 kernels sur M). Utile.
            │         │
            │         └── NON → Centered Alignment suffit.
            │
            └── Hardware quantique stable disponible ?
                      │
                      ├── OUI (2028+) → Kernels réels sur QPU.
                      │
                      └── NON → Simulation classique. Coût élevé.
```

### Quand investir dans le quantique (pour QMKL) ?

1. **Immédiatement :** Formulation QUBO pour sélection de kernels ($M > 12$)
2. **2027 :** Recuit quantique D-Wave pour QUBO ($M > 25$)
3. **2028 :** Circuits peu bruités → kernels réels compétitifs
4. **2030+ :** Données naturellement quantiques → avantage structurel prouvé


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# ── Arbre de décision visuel ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14), ax.set_ylim(0, 9)
ax.axis('off')

def node(ax, x, y, text, color, width=2.5, height=0.6, fontsize=9):
    box = FancyBboxPatch((x-width/2, y-height/2), width, height,
                          boxstyle='round,pad=0.1', facecolor=color,
                          edgecolor='grey', lw=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color='white' if color not in ['#f9f9f9','#fff9c4','#d5f5e3'] else '#333',
            wrap=True)

def arrow(ax, x1, y1, x2, y2, label='', color='#555'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.1, my, label, fontsize=8, color=color, fontweight='bold')

# Racine
node(ax, 7, 8.3, 'Mes données', '#2c3e50', width=2.5)

# Question 1 : N
node(ax, 7, 7.2, 'N > 1000 ?', '#7f8c8d', width=2.5)
arrow(ax, 7, 8.0, 7, 7.5)

# OUI → classique
node(ax, 11, 7.2, 'RBF-SVM
ou Random Forest
(non, QMKL trop lent)', '#c0392b', width=3, height=0.8)
arrow(ax, 8.25, 7.2, 9.5, 7.2, 'OUI')

# NON → Q2
node(ax, 7, 6.0, 'Structure périodique
ou interactions x_i·x_j ?', '#7f8c8d', width=3.5)
arrow(ax, 7, 6.95, 7, 6.3, 'NON')

# OUI → ZZ
node(ax, 11, 6.0, 'Essayer K_ZZ
QMKL peut aider', '#27ae60', width=3, height=0.7)
arrow(ax, 8.75, 6.0, 9.5, 6.0, 'OUI')

# NON → Q3
node(ax, 7, 4.8, 'Interprétabilité
de la sélection ?', '#7f8c8d', width=3.5)
arrow(ax, 7, 5.65, 7, 5.1, 'NON')

# OUI → QUBO
node(ax, 11, 4.8, 'QUBO
(sélectionne 2 kernels sur M)
Utile et interprétable', '#2980b9', width=3.3, height=0.8)
arrow(ax, 8.75, 4.8, 9.35, 4.8, 'OUI')

# NON → Centered
node(ax, 7, 3.6, 'Hardware QPU stable ?', '#7f8c8d', width=3.0)
arrow(ax, 7, 4.45, 7, 3.9, 'NON')

node(ax, 3, 3.6, 'Centered Alignment
(robuste, analytique)', '#1abc9c', width=3.0, height=0.7)
arrow(ax, 5.5, 3.6, 4.5, 3.6, 'NON')

node(ax, 11, 3.6, 'Kernels réels
sur QPU (2028+)', '#8e44ad', width=3.0, height=0.7)
arrow(ax, 8.5, 3.6, 9.5, 3.6, 'OUI')

# Légende
legend_items = [
    (mpatches.Patch(color='#c0392b'), 'Ne pas utiliser QMKL'),
    (mpatches.Patch(color='#27ae60'), 'QMKL potentiellement utile'),
    (mpatches.Patch(color='#2980b9'), 'QUBO recommandé'),
    (mpatches.Patch(color='#1abc9c'), 'Centered Alignment recommandé'),
    (mpatches.Patch(color='#8e44ad'), 'Futur (2028+)'),
]
ax.legend(handles=[h for h,_ in legend_items], labels=[l for _,l in legend_items],
          loc='lower left', fontsize=8.5, framealpha=0.9)

ax.set_title('Guide de décision : faut-il utiliser QMKL ?
'
             '(basé sur nos expériences et la littérature)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 4 — Perspectives 2026–2030

### Timeline des jalons techniques nécessaires

**2026 (maintenant) :** Simulation classique uniquement, $Q \leq 8$, $N \leq 300$.
- QMKL est un outil de recherche, pas de production.
- QUBO sur D-Wave cloud : fonctionnel mais $M$ limité.

**2027 :** Meilleure correction d'erreurs partielles (IBM Falcon, Heron r2+).
- Circuits plus profonds possibles ($\text{reps} = 2$).
- Kernels ZZFeatureMap sur hardware moins bruités.
- QAOA pour QUBO : viable pour $M \leq 50$.

**2028 :** Correction d'erreurs quantiques (QEC) partielle.
- $Q \leq 20$ qubits logiques en pratique.
- QMKL compétitif sur des datasets "quantique-friendly".
- Break-even avec le classique sur certains domaines.

**2030+ :** QEC complète, milliers de qubits logiques.
- Données intrinsèquement quantiques (chimie, physique des matériaux).
- Avantage quantique prouvé pour ces domaines.
- Finance : avantage incertain (données classiques).

### Ce qui changerait les résultats

1. **Données financières intrinsèquement quantiques** : si les prix d'actifs avaient une structure quantique naturelle (peu probable).
2. **Certifiability** : si on pouvait prouver qu'un kernel quantique spécifique est classiquement difficile à simuler pour notre domaine.
3. **Encodage adaptatif** : feature maps apprises (variational) qui maximisent l'alignment sur les données financières.
4. **Scalabilité** : approximation Nyström quantique rendant QMKL viable pour $N = 10\,000$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Roadmap temporelle ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(2025.5, 2031)
ax.set_ylim(0, 4)
ax.axis('off')

years = [2026, 2027, 2028, 2029, 2030, 2031]
ax.axhline(2, color='#2c3e50', lw=3)
for yr in years:
    ax.plot(yr, 2, 'o', color='#2c3e50', ms=12, zorder=3)
    ax.text(yr, 1.7, str(yr), ha='center', fontsize=10, fontweight='bold')

milestones = [
    (2026.2, 'Maintenant
• Q≤8, N≤300
• QUBO classique
• Simulation seule',
     '#3498db', 3.2),
    (2027.2, '2027
• Circuits plus profonds
• QAOA M≤50
• IBMHeron r2+',
     '#9b59b6', 3.2),
    (2028.2, '2028
• QEC partielle
• Q≤20 logiques
• Break-even ?',
     '#e67e22', 3.2),
    (2030.2, '2030+
• QEC complète
• Données quantiques
• Avantage prouvé ?',
     '#27ae60', 3.2),
]

for x, txt, col, y in milestones:
    ax.plot([x, x], [2.15, y-0.3], '--', color=col, lw=1.5, alpha=0.7)
    box = mpatches.FancyBboxPatch((x-0.35, y-0.1), 0.7, 0.9,
                                   boxstyle='round,pad=0.05',
                                   facecolor=col, alpha=0.15, edgecolor=col, lw=1.5)
    ax.add_patch(box)
    ax.text(x, y+0.35, txt, ha='center', va='bottom', fontsize=8,
            color='#2c3e50', bbox=dict(boxstyle='round', facecolor='white', edgecolor=col, alpha=0.9))

# Indicateurs sous la timeline
indicators = [
    (2026.2, 0.9, 'Q=4
N≤300', '#3498db'),
    (2027.2, 0.9, 'Q=6
N≤500', '#9b59b6'),
    (2028.2, 0.9, 'Q=10
N≤1000', '#e67e22'),
    (2030.2, 0.9, 'Q=50
N≤10000', '#27ae60'),
]
for x, y, txt, col in indicators:
    ax.text(x, y, txt, ha='center', va='center', fontsize=8, color=col,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=col, alpha=0.8))

ax.text(2028.5, 0.3, '← Aujourd'hui, QMKL = outil de recherche — Futur, QMKL potentiellement productif →',
        ha='center', fontsize=9, style='italic', color='#7f8c8d')
ax.set_title('Roadmap QMKL 2026–2031 : jalons techniques nécessaires
'
             'Les dates sont des estimations basées sur les roadmaps IBM, Google, IonQ',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 5 — Les 5 vérités inconfortables sur QMKL

Ce notebook se termine par une analyse critique honnête. Voici les 5 conclusions difficiles à accepter mais importantes à comprendre.

---

### Vérité 1 : QMKL ne bat pas les classiques sur les données financières en 2026

**Résultats empiriques :**
- German Credit : RBF-SVM $0.800$ > QMKL-Centered $0.743$ → **-7 pts**
- Bank Marketing : RBF-SVM $0.875$ > QMKL $0.861$ → **-1.4 pts**
- Breast Cancer : RBF-SVM $0.997$ > QMKL $0.993$ → **-0.4 pts**

Ce n'est pas un échec de l'implémentation — c'est la réalité du quantique en 2026 sur des données tabulaires classiques.

---

### Vérité 2 : QMKL-Boost (QUBO) bat Average mais pas le meilleur classique

QUBO améliore de **+4.1 pts** sur German Credit par rapport à Average QMKL. Mais reste encore -2 pts sous RBF-SVM.
La contribution réelle est l'**interprétabilité** (2 kernels sélectionnés = modèle explicable) et la **vitesse d'inférence** (2× moins de calcul).

---

### Vérité 3 : Les barren plateaus limitent l'expressivité à grand Q

Au-delà de $Q = 8$, le kernel perd >60% de variance → les données deviennent indiscernables.
L'hypothèse "espace de Hilbert exponentiel = avantage" ne se concrétise pas en pratique à cause de ce phénomène.

---

### Vérité 4 : Le hardware IBM Torino actuel est trop bruité

ZZ α=4.0 sur IBM Torino : AUC < 0.5 (pire que le hasard).
Le bruit des portes CNOT (~0.5% d'erreur) s'accumule exponentiellement avec la profondeur du circuit.

---

### Vérité 5 : La valeur actuelle de QMKL est dans la formulation, pas l'exécution

**Ce qui a de la valeur aujourd'hui :**
1. La formulation QUBO est directement transposable sur hardware quantique futur
2. L'analyse Shapley identifie les kernels importants (interprétabilité)
3. L'étude des barren plateaus guide le choix de Q pour les futurs QPU
4. Le framework MKL est agnostique au type de kernel — il fonctionnera aussi avec des kernels classiques

**Ce qui n'a pas encore de valeur pratique :**
- Les kernels quantiques simulés vs RBF-SVM sur données financières
- Le matériel NISQ actuel pour l'inférence en production

---

### Conclusion finale

QMKL est un framework **prometteur** mais **prématuré** pour la finance en 2026.
Sa valeur réside dans :
1. La préparation d'une infrastructure prête pour l'ère quantique
2. L'exploration systématique de l'espace des kernels via QUBO
3. La compréhension des limites (barren plateaus) qui guident la recherche future

**Recommandation :** Continuer la recherche, utiliser QUBO pour la sélection de kernels classiques aussi, et réévaluer quand les QPU auront des taux d'erreur < 0.1% par porte.
